In [53]:
# Helper Functions

import os
import math
import copy
import open3d as o3d
import numpy as np
from scipy.spatial.transform import Rotation as R
import pandas as pd
import re



def get_rotation_matrix_z(deg):
    """Creates a 3x3 rotation matrix for the Z-axis."""
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0], 
                     [s,  c, 0], 
                     [0,  0, 1]])


def merge_multiview_scan(data_dir, initial_pos, initial_angle, box_size):
    """Merges multiple view PCDs and removes points below the detected plane."""
    
    pcd_files = [f for f in os.listdir(data_dir) if f.lower().endswith('.pcd')]

    def extract_number(filename):
        numbers = re.findall(r'\d+', filename)
        return int(numbers[0]) if numbers else 0
    
    pcd_files.sort(key=extract_number)
    
    merged_pcd = o3d.geometry.PointCloud()
    
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos), 
        R=get_rotation_matrix_z(-initial_angle), 
        extent=np.array(box_size)
    )

    test = 0   

    print(f"Found {len(pcd_files)} PCD files. Processing...")

    for file_name in pcd_files:
        pcd = o3d.io.read_point_cloud(os.path.join(data_dir, file_name))
   
        # 1. Detect the plane
        plane_model, inliers = pcd.segment_plane(distance_threshold=3.0, ransac_n=3, num_iterations=2000)
        [a, b, c, d] = plane_model

        # 2. Extract all points as a numpy array
        pts = np.asarray(pcd.points)

        # 3. Calculate distance to plane for every point: ax + by + cz + d
        # Points on the plane result in 0. Points above are positive, below are negative.
        distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
        
        # 4. Create an index of points that are ABOVE the plane
        # A small offset (e.g., 0.5mm) to avoid keeping table noise
        above_plane_indices = np.where(distances > 0.5)[0]
        pcd = pcd.select_by_index(above_plane_indices)


        # 5. Crop to the OBB and merge
        if test == 0:            
            # create frame
            frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=20)
            frame.transform(T_base2ob_yolo)

            o3d.visualization.draw_geometries([pcd, obb, frame], window_name=f"Filtered Cloud - {file_name}")
            test += 1

        merged_pcd += pcd.crop(obb)

    # frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=20)
    # frame.transform(T_base2ob_yolo)
    # o3d.visualization.draw_geometries([merged_pcd, obb, frame], window_name=f"Filtered Cloud - {file_name}")

    return merged_pcd



def process_point_cloud(pcd, initial_pos, initial_angle, box_size):
    """Processes a single PCD file: removes ground plane and crops to OBB."""
    distance_threshold = 3.0 # mm, for RANSAC plane segmentation
    ransac_n = 3
    num_iterations = 2000
    plane_offset = 0.5 # mm, to ensure we keep points just above the plane

    # 1. Load the specific file    
    if pcd.is_empty():
        print(f"Warning: Point cloud is empty or not found.")
        return pcd

    # 2. Define the Oriented Bounding Box (OBB)
    # Using your rotation logic to align with the YOLO/Initial guess
    rot_matrix = get_rotation_matrix_z(-initial_angle) 
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos), 
        R=rot_matrix, 
        extent=np.array(box_size)
    )

    # 3. Detect and remove the ground plane (RANSAC)
    plane_model, inliers = pcd.segment_plane(distance_threshold=distance_threshold, 
                                             ransac_n=ransac_n, 
                                             num_iterations=num_iterations)
    [a, b, c, d] = plane_model

    # 4. Filter points above the plane
    pts = np.asarray(pcd.points)
    # ax + by + cz + d > offset
    distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
    above_plane_indices = np.where(distances > plane_offset)[0] 
    pcd_filtered = pcd.select_by_index(above_plane_indices)
    # 5. Crop to the region of interest
    # o3d.visualization.draw_geometries([final_pcd, obb], window_name="Downsampled Clouds with Normals")
    final_pcd = pcd_filtered.crop(obb)

    return final_pcd


def load_viewpoint_data(actual_dir, sim_dir):
    """
    Loads all actual and simulated point clouds automatically.

    Args:
        actual_dir (str): Path to 'pcd_data' folder
        sim_dir (str): Path to 'viewpoints_candidate' folder

    Returns:
        target_clouds (list): list of actual point clouds
        source_clouds (list): list of simulated point clouds
    """

    def extract_number(filename):
        numbers = re.findall(r'\d+', filename)
        return int(numbers[0]) if numbers else 0

    target_clouds = []
    source_clouds = []


    # find all actual viewpoint files
    actual_files = [f for f in os.listdir(actual_dir) if f.endswith(".pcd")]
    actual_files.sort(key=extract_number)


    for file in actual_files:

        print(f"Processing: {file}") # This will show you the culprit

        # extract index from filename (view00.pcd -> 0)
        index = int(file.replace("view", "").replace(".pcd", ""))

        sim_name = f"viewpoint_simulated_{index}.pcd"

        actual_path = os.path.join(actual_dir, file)
        sim_path = os.path.join(sim_dir, sim_name)

        try:
            actual_pcd = o3d.io.read_point_cloud(actual_path)
            sim_pcd = o3d.io.read_point_cloud(sim_path)

            target_clouds.append(actual_pcd)
            source_clouds.append(sim_pcd)

        except Exception as e:
            print(f"Error loading index {index}: {e}")

    return target_clouds, source_clouds


def pose_to_matrix(pose):
    """
    Converts [x, y, z, roll, pitch, yaw] in degrees to a 4x4 transformation matrix.
    If zyz=True, assumes the input is in ZYZ Euler Angles, otherwise XYZ Euler Angles.
    
    Args:
    - pose (list or array): [x, y, z, roll, pitch, yaw] in degrees.
    - zyz (bool): If True, interprets the last three values as ZYZ Euler angles. 
                  If False, uses XYZ Euler angles (default).
    
    Returns:
    - T (ndarray): The 4x4 homogeneous transformation matrix.
    """
    x, y, z = pose[:3]   # Translation vector
    roll, pitch, yaw = pose[3:]  # Rotation angles in degrees
    
    # Convert XYZ Euler Angles to a 3x3 rotation matrix
    rot_matrix = R.from_euler('xyz', [roll, pitch, yaw], degrees=True).as_matrix()

    # Build the 4x4 Homogeneous Transformation Matrix
    T = np.eye(4)  # Start with the identity matrix (4x4)
    T[:3, :3] = rot_matrix  # Set the upper-left 3x3 part to the rotation matrix
    T[:3, 3] = [x, y, z]    # Set the upper-right 3x1 part to the translation vector
    return T


def preprocess_normal(pcd, num_points=False, invert_normals=False):
    """Downsamples, estimates normals, and computes FPFH features."""
    if num_points:
        pcd_down = pcd.farthest_point_down_sample(num_points)
    else:
        pcd_down = pcd
    avg_dist = np.mean(pcd_down.compute_nearest_neighbor_distance())
    
    # Estimate Normals
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 2, max_nn=30))
    
    # Orientation fix: Ensure normals point 'up' (positive Z)
    normals = np.asarray(pcd_down.normals)
    
    if invert_normals:
        for i in range(len(normals)):
            if normals[i][2] < 0:
                normals[i] *= -1

    # Compute FPFH (Feature descriptors for global matching)
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down, o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 5, max_nn=100))
    
    return pcd_down, fpfh


def run_global_registration(source, target, source_fpfh, target_fpfh, voxel_size):
    """
    Performs RANSAC-based global registration to find a rough alignment.
    """
    distance_threshold = voxel_size * 1.5
    
    # RANSAC based on feature matching
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source, target, source_fpfh, target_fpfh, 
        mutual_filter=True,
        max_correspondence_distance=distance_threshold,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=3, 
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
        ], 
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
    )
    return result


def run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh):
    """
    Performs iterative RANSAC-based global registration to find the best rough alignment.
    Matches the logic of testing multiple thresholds to find the highest fitness and lowest RMSE.
    """
    max_attempts = 5
    best_fitness = -0.1
    best_inlier_rmse = 100.0
    best_result = None
    best_threshold = None 

    # Thresholds to test, from coarse (10.0) to fine (3.0)
    thresholds = [10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]
    # thresholds = [7]

    for attempt in range(max_attempts):
        for thr in thresholds:            
            result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
                source_down, target_down, source_fpfh, target_fpfh, 
                mutual_filter=True, # Improved matching
                max_correspondence_distance=thr,
                estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
                ransac_n=3, 
                checkers=[
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.85),
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(thr)
                ], 
                criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(10000, 0.99)
            )

            # Update best result if fitness is good and RMSE is lower
            if result.fitness > 0.85 and result.inlier_rmse < best_inlier_rmse:
                best_fitness = result.fitness
                best_inlier_rmse = result.inlier_rmse
                best_result = result
                best_threshold = thr
            
            # Early exit if we find a very high-quality match
            if best_fitness > 0.95 and best_inlier_rmse < 2.35: 
                print(f"Excellent Global Fit Found at Threshold {best_threshold}")
                return best_result, best_threshold
            
    if best_result is None:
        print("Warning: RANSAC could not find a fit above 0.85 fitness.")
        # Fallback to the last result generated if nothing met the 0.85 criteria
        return result, thr

    print(f"RANSAC Finished. Best Threshold: {best_threshold} | Fitness: {best_fitness:.4f}")
    return best_result, best_threshold


# One time local refinement using ICP
def run_local_refinement(source, target, initial_transformation, voxel_size):
    """
    Performs ICP registration to refine the alignment found by RANSAC.
    """
    # We use a smaller threshold for ICP to ensure high precision
    distance_threshold = voxel_size * 0.4
    
    result = o3d.pipelines.registration.registration_icp(
        source, target, distance_threshold, initial_transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
    )
    return result


# Progressive Local Refinement using ICP
def run_local_refinement_adaptive(source, target, initial_trans=None, best_ransac_thr=10):
    """
    Refines alignment using an iterative ICP loop. 
    Starts with a coarse threshold and progressively tightens for precision.
    """

    if initial_trans is None:
        initial_trans = np.eye(4)
        
    # Define a range of multipliers to tighten the search radius
    # multipliers = [1.0, 0.8, 0.5, 0.25]
    multipliers = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
    # multipliers = [0.6]
    thresholds = [best_ransac_thr * m for m in multipliers]
    
    best_result = None
    best_inlier_rmse = float('inf') # Start with the highest possible error
    
    print(f"{'Threshold':<12} | {'Fitness':<12} | {'RMSE':<12}")
    print("-" * 45)

    for thr in thresholds:
        # Execute Point-to-Point ICP
        reg_icp = o3d.pipelines.registration.registration_icp(
            source, target, thr, initial_trans,
            o3d.pipelines.registration.TransformationEstimationPointToPoint(),
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=1000)
        )

        # Logging the current iteration results
        print(f"{thr:<12.2f} | {reg_icp.fitness:<12.4f} | {reg_icp.inlier_rmse:<12.4f}")

        # Selection Logic: Prioritize lowest RMSE (highest precision) 
        # as long as the fitness is acceptable (e.g., > 85%)
        if reg_icp.fitness > 0.85 and reg_icp.inlier_rmse < best_inlier_rmse:
            best_inlier_rmse = reg_icp.inlier_rmse
            best_result = reg_icp

    # Fallback: if no run met the 85% fitness, return the last result
    return best_result if best_result is not None else reg_icp

In [ ]:
# Create a initial alignment from every viewpoint

# --- 1. Configuration ---
NUMBER_OF_POINTS = 10000
EXPERIMENT = "training_second_square_flange"
SOURCE_PATH = "workpiece/square_flange.stl" #  CAD STL model

DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}" # Multiview scans
SOURCE_DIR = f"viewpoints_candidate/testing_data/{EXPERIMENT}"

tf_obj = np.load(os.path.join(DATA_DIR, 'initial_obj_pose.npy'))
T_base2ob_yolo = np.load(os.path.join(DATA_DIR, 'T_base2ob_yolo.npy'))

# T_base2ob_yolo[1,3] += 5
# T_base2ob_yolo[2,3] -= 15

YOLO_POS = tf_obj[:3]                                   # Initial guess from YOLO
YOLO_ANGLE = tf_obj[4]                                  # Initial angle from YOLO
# CROP_BOX = [85, 65, 75]                               # ROI Obj1
CROP_BOX = [110, 90, 85]                               # ROI Obj31
CROP_BOX = [75, 75, 60]                               # ROI square_flange

# YOLO_POS[1] += 5 # buat kebutuhan motong

# --- 2. Data Preparation ---
mesh = o3d.io.read_triangle_mesh(SOURCE_PATH)
mesh.compute_vertex_normals()
source_cloud = mesh.sample_points_uniformly(number_of_points=NUMBER_OF_POINTS)
full_target_cloud = merge_multiview_scan(DATA_DIR, YOLO_POS, YOLO_ANGLE, CROP_BOX)

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0, origin=[0, 0, 0])
o3d.visualization.draw_geometries([full_target_cloud, mesh, world_frame], window_name="Merged Target Cloud")


#temp
T_extra = np.eye(4)

# Define 90 degrees around Z (adjust axis as needed: 'x', 'y', or 'z')
r = R.from_euler('x', 0, degrees=True)
T_extra[:3, :3] = r.as_matrix()
T_initial_guess = T_base2ob_yolo @ T_extra
#temp

# 3. Preprocess Normal
# source_cloud_transformed = copy.deepcopy(source_cloud)
source_cloud_transformed = copy.deepcopy(source_cloud).transform(T_initial_guess)
source_down, source_fpfh = preprocess_normal(source_cloud_transformed)
target_down, target_fpfh = preprocess_normal(full_target_cloud, num_points=10000, invert_normals=True)

source_down.paint_uniform_color([1, 0, 0])
target_down.paint_uniform_color([0, 0.651, 0.929])
o3d.visualization.draw_geometries([target_down, source_down], window_name="Processed Target Cloud")
# o3d.visualization.draw_geometries([target_down], window_name="Processed Target Cloud")


In [51]:
o3d.visualization.draw_geometries([source_cloud_transformed, world_frame], window_name="Merged Target Cloud")


In [35]:
# --- 3. Global Alignment (RANSAC) ---
print("Step 2: Running RANSAC Global Registration...")
ransac_res, best_thr = run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh)
print(ransac_res)

# Visualize RANSAC result
source_temp = copy.deepcopy(source_down)
source_temp.transform(ransac_res.transformation)
source_temp.paint_uniform_color([1, 0, 0])
target_down.paint_uniform_color([0, 0.651, 0.929])
o3d.visualization.draw_geometries([source_temp, target_down], window_name="RANSAC Result")

Step 2: Running RANSAC Global Registration...
[Open3D WARNING] Too few correspondences (38) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (38) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (38) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (38) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (38) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (38) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (38) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (38) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (38) after mutual filter, fall back to original correspondences.


In [54]:
# --- 4. Local Alignment (ICP) ---
print("Step 3: Running ICP Local Refinement...")
# icp_res = run_local_refinement_adaptive(source_down, target_down, ransac_res.transformation, best_thr)
icp_res = run_local_refinement_adaptive(source_down, target_down)

print(icp_res)

# --- 5. Extract Final Results ---
fine_correction_transformation = icp_res.transformation # just the final ICP result
# merge_full_transformation = fine_correction_transformation
merge_full_transformation = np.dot(fine_correction_transformation, T_initial_guess) # Combine the initial YOLO-based transformation with the final ICP correction

initial_POS = merge_full_transformation[:3, 3] 
initial_ORI = merge_full_transformation[:3, :3] 
print("="*30)
print(f"Final Position: {initial_POS}")
print(f"Final Orientation Matrix:\n{initial_ORI}")
print(f"Fitness: {icp_res.fitness:.4f}")
print(f"RMSE: {icp_res.inlier_rmse:.4f}")


# --- 6. Final Visualization ---
source_final = copy.deepcopy(source_cloud).transform(merge_full_transformation)
source_final.paint_uniform_color([1, 0, 0])         # Red: CAD Model
target_cloud.paint_uniform_color([0, 0.65, 0.93])   # Blue: Scanned Data
o3d.visualization.draw_geometries([source_final, target_cloud])

# Save the final transformation for use in the evaluation step
FULL_TRANSFORM_SAVE_DIR = f'evaluation_result/{EXPERIMENT}/merge_full_transformation.npy'
np.save(FULL_TRANSFORM_SAVE_DIR, merge_full_transformation)

Step 3: Running ICP Local Refinement...
Threshold    | Fitness      | RMSE        
---------------------------------------------
10.00        | 1.0000       | 1.3379      
9.00         | 1.0000       | 1.3379      
8.00         | 1.0000       | 1.3379      
7.00         | 1.0000       | 1.3379      
6.00         | 1.0000       | 1.3379      
5.00         | 1.0000       | 1.3379      
4.00         | 0.9986       | 1.3292      
3.00         | 0.9937       | 1.3112      
2.00         | 0.9550       | 1.2540      
1.00         | 0.2662       | 0.7673      
RegistrationResult with fitness=9.550000e-01, inlier_rmse=1.253981e+00, and correspondence_set size of 9550
Access transformation to get result.
Final Position: [627.9841012   28.34268482   9.29814333]
Final Orientation Matrix:
[[ 0.00891653 -0.99976536 -0.01974158]
 [ 0.99990729  0.0091175  -0.01011321]
 [ 0.01029083 -0.01964958  0.99975397]]
Fitness: 0.9550
RMSE: 1.2540


In [ ]:
# empty #

In [8]:
target_clouds, source_clouds = load_viewpoint_data(DATA_DIR, SOURCE_DIR)


Processing: view0.pcd
Processing: view1.pcd
Processing: view2.pcd
Processing: view3.pcd
Processing: view4.pcd
Processing: view5.pcd
Processing: view6.pcd
Processing: view7.pcd
Processing: view8.pcd
Processing: view9.pcd
Processing: view10.pcd
Processing: view11.pcd
Processing: view12.pcd
Processing: view13.pcd
Processing: view14.pcd
Processing: view15.pcd
Processing: view16.pcd
Processing: view17.pcd
Processing: view18.pcd
Processing: view19.pcd
Processing: view20.pcd
Processing: view21.pcd
Processing: view22.pcd
Processing: view23.pcd
Processing: view24.pcd
Processing: view25.pcd
Processing: view26.pcd
Processing: view27.pcd
Processing: view28.pcd
Processing: view29.pcd
Processing: view30.pcd
Processing: view31.pcd
Processing: view32.pcd
Processing: view33.pcd
Processing: view34.pcd
Processing: view35.pcd
Processing: view36.pcd
Processing: view37.pcd
Processing: view38.pcd
Processing: view39.pcd
Processing: view40.pcd
Processing: view41.pcd
Processing: view42.pcd
Processing: view43.pc

In [62]:
# Configuration and Directories
PROCESSED_SAVE_DIR = f"processed_data/{EXPERIMENT}/"
os.makedirs(PROCESSED_SAVE_DIR, exist_ok=True)

merge_full_transformation = np.load(f'evaluation_result/{EXPERIMENT}/merge_full_transformation.npy')

# Data preparation
# target_clouds, source_clouds = load_viewpoint_data(DATA_DIR, SOURCE_DIR)
results_list = []


# For Testing one viewpoint
idx = 123
# o3d.visualization.draw_geometries([target_clouds[idx]]) ## TWO ## from loader

# target_cloud = process_point_cloud(target_clouds[idx], YOLO_POS, YOLO_ANGLE, CROP_BOX)
# target_cloud , _ = preprocess_normal(target_cloud, invert_normals=True)
# source_cloud_transformed = copy.deepcopy(source_clouds[idx]).transform(merge_full_transformation)

# source_cloud_transformed.paint_uniform_color([1, 0, 0])
# target_cloud.paint_uniform_color([0, 0.651, 0.929])
# o3d.visualization.draw_geometries([target_cloud, source_cloud_transformed])

## o3d.io.write_point_cloud(f"processed_data/view{idx}.pcd", target_cloud)


# Full alignment on each viewpoint
for i in range(len(target_clouds)):
    # Process the real scan (Crop & Clean)
    target_cloud = process_point_cloud(target_clouds[i], YOLO_POS, YOLO_ANGLE, CROP_BOX)
    target_cloud , _ = preprocess_normal(target_cloud, invert_normals=True)
    
    # Transform the CAD (Source) to the expected pose
    source_cloud_transformed = copy.deepcopy(source_clouds[i]).transform(merge_full_transformation)

    # 1. Save processed target_cloud to .pcd
    filename = f"viewpoint_simulated_{i}.pcd"
    save_path = os.path.join(PROCESSED_SAVE_DIR, filename)
    o3d.io.write_point_cloud(save_path, target_cloud)

    # 2. Compute Metrics
    # Distance from CAD to Real Scan
    dist_s2r_array = np.asarray(source_cloud_transformed.compute_point_cloud_distance(target_cloud))
    dist_s2r = dist_s2r_array.mean()
    
    # Distance from Real Scan to CAD
    dist_r2s_array = np.asarray(target_cloud.compute_point_cloud_distance(source_cloud_transformed))
    dist_r2s = dist_r2s_array.mean()
    
    # Main Regression Targets
    chamfer_value = dist_s2r + dist_r2s
    asymmetry_value = abs(dist_s2r - dist_r2s)

    print(f"Saving viewpoint {i}:")
    # 3. Store results for CSV
    results_list.append({
        'filename': filename,
        'dist_s2r': dist_s2r,
        'dist_r2s': dist_r2s,
        'chamfer_value': chamfer_value,
        'asymmetry_value': asymmetry_value
    })

# --- 4. Save Metadata ---
csv_path = os.path.join(PROCESSED_SAVE_DIR, "metadata.csv")
df = pd.DataFrame(results_list)
df.to_csv(csv_path, index=False)

print(f"\nEvaluation Complete!")
print(f"PCD files and metadata saved to: {PROCESSED_SAVE_DIR}")

Saving viewpoint 0:
Saving viewpoint 1:
Saving viewpoint 2:
Saving viewpoint 3:
Saving viewpoint 4:
Saving viewpoint 5:
Saving viewpoint 6:
Saving viewpoint 7:
Saving viewpoint 8:
Saving viewpoint 9:
Saving viewpoint 10:
Saving viewpoint 11:
Saving viewpoint 12:
Saving viewpoint 13:
Saving viewpoint 14:
Saving viewpoint 15:
Saving viewpoint 16:
Saving viewpoint 17:
Saving viewpoint 18:
Saving viewpoint 19:
Saving viewpoint 20:
Saving viewpoint 21:
Saving viewpoint 22:
Saving viewpoint 23:
Saving viewpoint 24:
Saving viewpoint 25:
Saving viewpoint 26:
Saving viewpoint 27:
Saving viewpoint 28:
Saving viewpoint 29:
Saving viewpoint 30:
Saving viewpoint 31:
Saving viewpoint 32:
Saving viewpoint 33:
Saving viewpoint 34:
Saving viewpoint 35:
Saving viewpoint 36:
Saving viewpoint 37:
Saving viewpoint 38:
Saving viewpoint 39:
Saving viewpoint 40:
Saving viewpoint 41:
Saving viewpoint 42:
Saving viewpoint 43:
Saving viewpoint 44:
Saving viewpoint 45:
Saving viewpoint 46:
Saving viewpoint 47:
Sa